### Cleaning Plan
- Scores - drop score < 7.5 and score > 9.0
- Watchers - drop dramas with < 500 watchers
- Episodes - drop dramas with < 10 episodes
- Synopsis 
  - strip everything from '(Source: MyDramaList)' onward
  - after stripping, drop if synopsis < 100 chars  

In [1]:
import polars as pl
from pathlib import Path

RAW_PATH = Path("../data/raw/dramas.ndjson")
CLEAN_PATH = Path("../data/cleaned/dramas_clean.ndjson")

In [2]:
raw_df = pl.read_ndjson(RAW_PATH)
print(f"Raw dataset: {raw_df.height} dramas")

Raw dataset: 2694 dramas


In [11]:
df_clean = (
    raw_df.lazy()

    .filter(pl.col("episodes") >= 10)
    .filter(pl.col("watchers") >= 500)
    .filter(pl.col("mdl_score").is_between(7.5, 9.0))
    .filter(pl.col("genres").list.len() > 0)
    .filter(pl.col("tags").list.len() > 0)
    # synopsis cleaning
    .with_columns(
        pl.col("synopsis")
        .str.replace(r"(?i)\s*\(Source:.*?\)", "")     
        .str.replace(r"(?i)\s*Adapted from.*$", "")
        .str.replace_all(r"~+\s*", "")     
        .str.replace(r"Edit Translation.*$", "")        
        .str.replace_all(r"[\n\r\t]+", " ")             
        .str.strip_chars()                              
        .alias("synopsis")                              
    )
    
    .filter(pl.col("synopsis").str.len_chars() >= 100)
    # normalize lists for consistent DB filtering
    .with_columns(
        pl.col("genres").list.eval(pl.element().str.to_lowercase().str.strip_chars()),
        pl.col("tags").list.eval(pl.element().str.to_lowercase().str.strip_chars())
    )
    .collect()
)
print(f"Clean dataset: {df_clean.height} dramas")
print(f"Dropped: {raw_df.height - df_clean.height} dramas")

Clean dataset: 1437 dramas
Dropped: 1257 dramas


In [6]:
print(f"Score range: {df_clean['mdl_score'].min()} – {df_clean['mdl_score'].max()}")
print(f"Min episodes: {df_clean['episodes'].min()}")
print(f"Min watchers: {df_clean['watchers'].min()}")
print(f"Min synopsis length: {df_clean['synopsis'].str.len_chars().min()}")
print(f"Empty genres: {df_clean.filter(pl.col('genres').list.len() == 0).height}")
print(f"Empty tags: {df_clean.filter(pl.col('tags').list.len() == 0).height}")
print(f"Null synopses: {df_clean['synopsis'].is_null().sum()}")

Score range: 7.5 – 9.0
Min episodes: 10
Min watchers: 504
Min synopsis length: 111
Empty genres: 0
Empty tags: 0
Null synopses: 0


In [12]:
df_clean["synopsis"][1]

"In sixth-century China, the Emperor of Great Liang orders the unjust execution of his brother-in-law Marshal Lin Xie alongside the Lin family, his 70,000 army soldiers, and Crown Prince Qi. Secretly surviving the massacre is Lin Xie's son, Lin Shu, who undergoes medical treatment that changes his appearance entirely and leaves him in a weakened state, unable to ever perform martial arts again. Lin Shu changed his name to Mei Chang Suand later became the chief of the pugilist world and established the Jiangzuo Alliance. Twelve years later, Mei Chang Su returns to the capital with a secret plan after being sought after by Prince Yu and Prince Xian during their fight for the throne. He decides to covertly assist Prince Jing, the unfavoured son of the Emperor, and wisely rids the court of all scheming officials."

In [8]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.write_ndjson(CLEAN_PATH)
print(f"Saved to {CLEAN_PATH}")

Saved to ..\data\cleaned\dramas_clean.ndjson
